# Lab 4.9 &mdash; Data Efficiency: How Much Training Data Do You Need?

**Level:** Intermediate &nbsp;|&nbsp; **Est. time:** 30 min &nbsp;|&nbsp; **Day 2 &middot; Module 4 &mdash; Pre-trained Models & Fine-tuning**

### What you'll do
- Train the same head on growing fractions of data
- Plot a learning curve of validation accuracy vs train size
- See why transfer learning shines with little data

> **How this lab works (experiential flow):** read the **Concept**, run the **Demo** to see it work, then complete **Your Turn** by replacing every `___` placeholder. Run the **grader** cell at the end &mdash; it prints `[PASS]` / `[FAIL]` / `[TODO]` and a final `Score`. Aim for a full score.

**Reference:** [Module 4 slides &mdash; Transfer learning](../../presentation/day2-module4-pretrained-models-and-fine-tuning.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 4 labs](./index.html)

In [ ]:
# Setup -- run me first
import os
WORK = "/tmp/biaa-lab-04-09"
os.makedirs(WORK, exist_ok=True)
print("Working dir:", WORK)

## Concept
A big reason to use pre-trained models: they need **little task data**. We train the head on
growing slices of the training set and watch validation accuracy rise &mdash; a **learning
curve**. With frozen good features, even a few dozen examples go a long way.

In [ ]:
# DEMO
# A tiny labelled sentiment dataset (1 = positive, 0 = negative)
SENT = [
    ("i love this it is great", 1),
    ("a great and wonderful film", 1),
    ("truly wonderful i love it", 1),
    ("excellent and brilliant work", 1),
    ("the best most brilliant story", 1),
    ("i love how great it is", 1),
    ("wonderful excellent and great fun", 1),
    ("a brilliant and great success", 1),
    ("great fun i really love it", 1),
    ("the best film wonderful and brilliant", 1),
    ("excellent great and lovely work", 1),
    ("i love this brilliant great film", 1),
    ("wonderful great and the best", 1),
    ("so good i love it great", 1),
    ("i hate this it is terrible", 0),
    ("a terrible and awful film", 0),
    ("truly awful i hate it", 0),
    ("boring and terrible work", 0),
    ("the worst most boring story", 0),
    ("i hate how bad it is", 0),
    ("awful boring and dull mess", 0),
    ("a terrible and bad failure", 0),
    ("boring mess i really hate it", 0),
    ("the worst film awful and boring", 0),
    ("terrible bad and dull work", 0),
    ("i hate this awful boring film", 0),
    ("awful terrible and the worst", 0),
    ("so bad i hate it terrible", 0),
]
texts  = [t for t, y in SENT]
labels = [y for t, y in SENT]
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
Xtr_t, Xval_t, ytr, yval = train_test_split(texts, labels, test_size=0.3, random_state=0, stratify=labels)
vec = TfidfVectorizer().fit(Xtr_t)
Xtr, Xval = vec.transform(Xtr_t), vec.transform(Xval_t)
print("train pool:", Xtr.shape[0], "| val:", Xval.shape[0])

## Your Turn
For each fraction, train on the first k examples and record validation accuracy.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import numpy as np

def curve():
    n = Xtr.shape[0]
    accs = []
    for frac in [0.25, 0.5, 1.0]:
        k = max(2, int(n * frac))
        head = LogisticRegression(max_iter=1000)
        head.fit(Xtr[:k], ytr[:k])   # train on the first k examples
        acc = ___   # TODO: accuracy_score(yval, head.predict(Xval))
        accs.append(acc)
    return accs

try:
    accs = curve()
    print('val accuracy at 25%/50%/100%:', [round(a,3) for a in accs])
    import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
    plt.plot([0.25,0.5,1.0], accs, 'o-'); plt.xlabel('train fraction'); plt.ylabel('val accuracy')
    plt.title('Learning curve'); plt.savefig(WORK + '/learning_curve.png', dpi=90)
    print('saved', WORK + '/learning_curve.png')
except Exception as e: print('Fill the blanks, then re-run.', type(e).__name__)

In [ ]:
# === Auto-grader: run after filling the blanks above ===
_results = []
def _rec(label, status, extra=""):
    _results.append(status); print(f"[{status}] {label}" + (f" -- {extra}" if extra else ""))
def expect(label, got, want):
    if got == "___" or got is None: _rec(label, "TODO")
    elif got == want: _rec(label, "PASS")
    else: _rec(label, "FAIL", f"got {got!r}")
def expect_true(label, fn):
    try: _rec(label, "PASS" if fn() else "FAIL")
    except Exception as e: _rec(label, "TODO", type(e).__name__)

expect_true("curve returns 3 accuracies", lambda: len(curve()) == 3)
expect_true("all accuracies are valid probabilities", lambda: all(0.0 <= a <= 1.0 for a in curve()))
expect_true("full data is at least as good as the smallest slice", lambda: curve()[-1] >= curve()[0] - 1e-9)

_p = _results.count("PASS")
print(f"\nScore: {_p}/{len(_results)}")
print("All checks passed -- lab complete!" if _p == len(_results) else "Keep going: fill the blanks marked ___ and re-run.")

---
### Key takeaway
Transfer learning is **data-efficient**: good frozen features mean a small head learns from few labels. That is why fine-tuning beats training from scratch.

[&#8592; All Module 4 labs](./index.html) &nbsp;&middot;&nbsp; [Module 4 slides](../../presentation/day2-module4-pretrained-models-and-fine-tuning.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>